In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [59]:
df = pd.read_csv("csv/data.csv")
pd.set_option('display.max_columns', None)
# pd.reset_option('display.max_columns')
# Маржа = цена продажи - себестоимость

df = df.drop_duplicates()

df["margin"] = df["sale_price"] - df["cost"]
df["revenue"] = df["sale_price"]

df['date'] = pd.to_datetime(df["created_at"], errors='coerce', format='ISO8601')

df['return_flg'] = (~df['returned_at'].isna()).astype(int)
df_delivered = df[df['status'].str.strip().str.lower() == 'complete']
df_delivered_not_returned = df[(df['status'].str.strip().str.lower() == 'complete') & (df['returned_at'].isna())]
df_returned = df[~(df['returned_at'].isna())]

total_revenue = df_delivered_not_returned["revenue"].sum()
total_margin = df_delivered_not_returned["margin"].sum()


# abc сегменты

In [73]:
cutoff_date = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=365)
df_last = df_delivered_not_returned[df_delivered_not_returned['date'] >= cutoff_date].copy()

user_rev = df_last.groupby("user_id")["revenue"].sum().reset_index()
user_rev = user_rev.sort_values("revenue", ascending=False)

user_rev["cum_revenue"] = user_rev["revenue"].cumsum()
total_revenue = user_rev["revenue"].sum()
user_rev["cum_share"] = user_rev["cum_revenue"] / total_revenue


def abc_class(x):
    if x <= 0.8:
        return "A"
    elif x <= 0.95:
        return "B"
    else:
        return "C"

user_rev["ABC_segment"] = user_rev["cum_share"].apply(abc_class)

all_users = pd.DataFrame(df_delivered_not_returned['user_id'].unique(), columns=['user_id'])

# Соединяем полный список с результатами ABC (left join)
# Те, кого не было в df_recent, получат NaN в колонке revenue
df_abc_final = all_users.merge(
    user_rev[['user_id', 'revenue', 'ABC_segment']], 
    on='user_id', 
    how='left'
)
df_abc_final['ABC_segment'] = df_abc_final['ABC_segment'].fillna('C')

df_abc = df_abc_final[["user_id", "revenue", "ABC_segment"]]
df_abc.to_csv("csv/cluster/user_abc.csv", index=False)

In [74]:
total_length = df_abc.shape[0]
print(total_length)
for m in ['A', 'B', 'C']:
    l = df_abc[df_abc['ABC_segment'] == m].shape[0]
    print(m, ':', l, l/total_length)

27473
A : 5300 0.19291668183307248
B : 3492 0.12710661376624321
C : 18681 0.6799767044006844


выполняется принцип паретто

# RFM сегменты

In [53]:
snapshot_date = df["date"].max() + pd.Timedelta(days=1)

rfm = df_delivered_not_returned.groupby("user_id").agg(
    recency=("date", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("revenue", "sum")
).reset_index()

In [84]:

rfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['monetary'], q=5, labels=[1,2,3,4,5])

rfm["RFM_score"] = (
    rfm["R"].astype(str) +
    rfm["F"].astype(str) +
    rfm["M"].astype(str)
).astype(int)

def rfm_segment(row):
    if row['R'] >= 4 and row['F'] >= 4 and row['M'] >= 4:
        return 'loyal'
    elif row['R'] == 3 and row['F'] <= 3 and row['M'] <= 3:
        return 'inter'
    elif row['R'] == 2 and row['F'] <= 2 and row['M'] <= 3:
        return 'risk' 
    elif row['R'] == 1 and row['F'] <= 2 and row['M'] <= 2:
        return 'lost'
    else:
        return 'needs attention'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

rfm["RFM_sum"] = rfm["R"].astype(int) + rfm["F"].astype(int) + rfm["M"].astype(int)

In [85]:
rfm.to_csv("csv/cluster/user_rfm.csv", index=False)

In [79]:
total_l = rfm.shape[0]
for m in range(5):
    l = rfm[rfm['M'] == m+1].shape[0]
    print(f'M = {m+1}:', l, l/total_l)

print('')
for r in range(5):
    l = rfm[rfm['R'] == r+1].shape[0]
    print(f'R = {r+1}:', l, l/total_l)

print('')
for f in range(5):
    l = rfm[rfm['F'] == f+1].shape[0]
    print(f'F = {f+1}:', l, l/total_l)

M = 1: 5496 0.20005095912350307
M = 2: 5549 0.2019801259418338
M = 3: 5439 0.19797619480944928
M = 4: 5494 0.19997816037564153
M = 5: 5495 0.2000145597495723

R = 1: 5492 0.19990536162778
R = 2: 5487 0.19972336475812616
R = 3: 5496 0.20005095912350307
R = 4: 5471 0.19914097477523388
R = 5: 5527 0.2011793397153569

F = 1: 5495 0.2000145597495723
F = 2: 5494 0.19997816037564153
F = 3: 5495 0.2000145597495723
F = 4: 5494 0.19997816037564153
F = 5: 5495 0.2000145597495723
